# Re-run: the label effect L on six models (Paper VI, RQ2)

This notebook recomputes the headline of RQ2 from the per-case model outputs published with release
`ai-grader-reliability-2026`, with nothing else: no model call, no corpus, no repository code. Put
`per-case-model-outputs.jsonl` (or the `.gz` the report offers) beside this notebook, or set `RELEASE_DIR`.

**L** is the grade under the high attainment label minus the grade under the low label, on the same
answer: `trueLabel − swappedLabel` in the high tail and `swappedLabel − trueLabel` in the low tail,
averaged over every answer graded in all three arms (control, true label, swapped label).

In [1]:
# Finding the published data. This cell is the same in every notebook of this report.
import gzip, json, os, sys
from pathlib import Path

# Importing from the release's code/ folder must not leave __pycache__ folders in it.
sys.dont_write_bytecode = True

RELEASE_ID = "ai-grader-reliability-2026"   # filled in by board/render_report.py
DATA_NAME = "per-case-model-outputs.jsonl"


def find_release_file(name=DATA_NAME):
    """Look beside this notebook, in RELEASE_DIR, in the folders above, and in a repository checkout."""
    here = Path.cwd()
    places = [Path(os.environ["RELEASE_DIR"])] if os.environ.get("RELEASE_DIR") else []
    places += [here, *here.parents]
    places += [p / "research/wiki/research/releases" / RELEASE_ID for p in [here, *here.parents]]
    for d in places:
        for n in (name, name + ".gz"):
            if (d / n).is_file():
                return d / n
    raise FileNotFoundError(
        f"Could not find {name} (or {name}.gz). Put it in the same folder as this notebook: download it "
        f"from the report ('Download the data file', beside each notebook) or take it from the release "
        f"directory. Or set the environment variable RELEASE_DIR to the folder that holds it.")


def per_case(runs=None):
    """Yield the per-case records of the published file, optionally only for the named runs."""
    path = find_release_file()
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt", encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            if r.get("record") == "case" and (runs is None or r["run"] in runs):
                yield r


def code_dir():
    """The release's code package (its code/ folder): in RELEASE_DIR, beside this notebook or in a folder
    above it (as when the notebook sits in the unzipped release's notebooks/ folder), in a folder named
    after the release there, or in a repository checkout."""
    here = Path.cwd()
    places = [Path(os.environ["RELEASE_DIR"])] if os.environ.get("RELEASE_DIR") else []
    for p in [here, *here.parents]:
        places += [p, p / RELEASE_ID, p / "research/wiki/research/releases" / RELEASE_ID]
    for d in places:
        if (d / "code" / "SHA256SUMS").is_file():
            return d / "code"
    raise FileNotFoundError(
        f"Could not find the code package of release {RELEASE_ID} (its code/ folder). Download the release "
        f"from the report ('Download the release', in the sidebar or beside each notebook), unzip it, and "
        f"run this notebook from the unzipped folder or its notebooks/ folder. Or set the environment "
        f"variable RELEASE_DIR to the unzipped {RELEASE_ID} folder.")


def find_code_file(relpath):
    """A file inside the release's code package, by its path there (e.g. studies/.../out/result.json)."""
    path = code_dir() / relpath
    if not path.is_file():
        raise FileNotFoundError(f"The code package at {code_dir()} has no {relpath}: is it release {RELEASE_ID}?")
    return path


def load_code_json(relpath):
    """A committed output of the code package, parsed."""
    return json.loads(find_code_file(relpath).read_text(encoding="utf-8"))


def shown(path):
    """A path as printed: relative to this notebook's folder, so an executed notebook names no machine's folders."""
    return os.path.relpath(path, Path.cwd())


def corpus_file(env_var, name):
    """A public corpus file the reader has downloaded: $env_var/name, or name beside this notebook."""
    for d in ([Path(os.environ[env_var])] if os.environ.get(env_var) else []) + [Path.cwd()]:
        if (d / name).is_file():
            return d / name
    raise FileNotFoundError(f"This notebook also needs {name} from the public corpus. Download the corpus "
                            f"(its location and hash are in the release file), then set {env_var} to its "
                            f"folder or put {name} beside this notebook.")


print("Data:", shown(find_release_file()))

Data: per-case-model-outputs.jsonl


In [2]:
from collections import defaultdict
import numpy as np

RUNS = {"Gemini 3.5 Flash Lite": "grader-vi/swap-gemini", "GPT-4.1-mini": "grader-vi/swap-gpt41mini",
        "GPT-5-mini": "grader-vi/swap-gpt5mini", "Qwen3-235B": "grader-vi/swap-qwen",
        "Claude Opus 5.5": "breadth/swap-opus55", "Llama 4 Maverick": "breadth/swap-llama4"}
cases = defaultdict(list)
for r in per_case(set(RUNS.values())):
    cases[r["run"]].append(r)
{m: len(cases[run]) for m, run in RUNS.items()}

{'Gemini 3.5 Flash Lite': 300,
 'GPT-4.1-mini': 300,
 'GPT-5-mini': 300,
 'Qwen3-235B': 300,
 'Claude Opus 5.5': 300,
 'Llama 4 Maverick': 300}

In [3]:
def label_effect(rows):
    rows = [r for r in rows if None not in (r["control"], r["trueLabel"], r["swappedLabel"])]
    hi = np.array([r["tail"] == "high" for r in rows])
    t = np.array([r["trueLabel"] for r in rows], float)
    s = np.array([r["swappedLabel"] for r in rows], float)
    hl = np.where(hi, t - s, s - t)          # mark under the high label minus mark under the low label
    learner = np.array([str(r["learnerId"]) for r in rows])
    return hl, hi, learner

point = {}
for m, run in RUNS.items():
    hl, hi, _ = label_effect(cases[run])
    point[m] = (hl.mean(), hl[hi].mean(), hl[~hi].mean(), len(hl))
for m, (L, Lh, Ll, n) in point.items():
    print(f"{m:22s} L {L:+.2f}   high {Lh:+.2f}   low {Ll:+.2f}   (n = {n})")

Gemini 3.5 Flash Lite  L +0.37   high +0.42   low +0.32   (n = 300)
GPT-4.1-mini           L +0.38   high +0.50   low +0.26   (n = 298)
GPT-5-mini             L +0.52   high +0.57   low +0.47   (n = 297)
Qwen3-235B             L +0.75   high +0.79   low +0.72   (n = 290)
Claude Opus 5.5        L +0.28   high +0.30   low +0.25   (n = 300)
Llama 4 Maverick       L +0.54   high +0.51   low +0.56   (n = 300)


In [4]:
# The paper's printed values (cross-model table under RQ2). Point estimates must match exactly.
PRINTED = {"Gemini 3.5 Flash Lite": (0.37, 0.42, 0.32), "GPT-4.1-mini": (0.38, 0.50, 0.26),
           "GPT-5-mini": (0.52, 0.57, 0.47), "Qwen3-235B": (0.75, 0.79, 0.72),
           "Claude Opus 5.5": (0.28, 0.30, 0.25), "Llama 4 Maverick": (0.54, 0.51, 0.56)}
for m, want in PRINTED.items():
    got = tuple(round(x, 2) for x in point[m][:3])
    assert got == want, (m, got, want)
print("All 18 point estimates match the paper.")

All 18 point estimates match the paper.


In [5]:
# Intervals: a bootstrap that resamples students within each tail (the unit the label belongs to).
# Same scheme as the study, but not its random stream, so endpoints can differ in the last digit.
def learner_boot(hl, hi, learner, reps=4000, seed=20261005):
    rng = np.random.default_rng(seed)
    groups = {tail: [np.flatnonzero((learner == j) & (hi == tail)) for j in np.unique(learner[hi == tail])]
              for tail in (True, False)}
    out = np.empty(reps)
    for b in range(reps):
        idx = np.concatenate([g[k] for tail in (True, False)
                              for g in [groups[tail]] for k in rng.integers(0, len(g), len(g))])
        out[b] = hl[idx].mean()
    return np.percentile(out, [2.5, 97.5])

for m, run in RUNS.items():
    hl, hi, learner = label_effect(cases[run])
    lo, up = learner_boot(hl, hi, learner)
    print(f"{m:22s} L {hl.mean():+.2f}  [{lo:+.2f}, {up:+.2f}]")

Gemini 3.5 Flash Lite  L +0.37  [+0.30, +0.44]
GPT-4.1-mini           L +0.38  [+0.30, +0.45]
GPT-5-mini             L +0.52  [+0.44, +0.60]
Qwen3-235B             L +0.75  [+0.69, +0.82]
Claude Opus 5.5        L +0.28  [+0.23, +0.31]
Llama 4 Maverick       L +0.54  [+0.42, +0.65]
